# EDA ENFR · Parte 2: medidas estadísticas

Complemento del notebook `ENFR_EDA.ipynb`. Calcula, sobre los microdatos limpios que genera el ETL:

1. **Media y varianza**: descriptivas de las variables continuas (edad e IMC)
2. **Matriz de correlación** entre las cuatro condiciones, el IMC y la edad
3. **Odds ratio con intervalo de confianza del 95%** de factores asociados a la hipertensión
4. **Prevalencia de hipertensión estandarizada por edad** por provincia

Exporta `enfr_descriptivas.csv`, `enfr_correlacion.csv`, `enfr_odds_ratio.csv` y `enfr_estandarizada.csv`,
que alimentan la pestaña **Estadística** del dashboard.

> Todo se calcula sobre la edición **2018**. Al correrlo, comparar los resultados con los CSV ya entregados:
> si algún número difiere, revisar la definición de la variable en la celda correspondiente.

## 1 · Carga

In [ ]:
import pandas as pd, numpy as np, glob, os

ENTRADA = "/content/drive/MyDrive/ENFR/procesado"   # <- misma carpeta que usa ENFR_EDA.ipynb
SALIDA  = ENTRADA

archivos = sorted(glob.glob(os.path.join(ENTRADA, "enfr_*_limpio.csv")))
if archivos:
    df = pd.concat([pd.read_csv(a) for a in archivos], ignore_index=True)
else:
    df = pd.read_parquet(os.path.join(ENTRADA, "enfr_armonizado.parquet"))

COND = ["hta", "diabetes", "colesterol_alto", "obesidad"]
NOMBRE = {"hta": "Hipertensión", "diabetes": "Diabetes", "colesterol_alto": "Colesterol alto", "obesidad": "Obesidad"}
PROV = {2:"CABA",6:"Buenos Aires",10:"Catamarca",14:"Córdoba",18:"Corrientes",22:"Chaco",
        26:"Chubut",30:"Entre Ríos",34:"Formosa",38:"Jujuy",42:"La Pampa",46:"La Rioja",
        50:"Mendoza",54:"Misiones",58:"Neuquén",62:"Río Negro",66:"Salta",70:"San Juan",
        74:"San Luis",78:"Santa Cruz",82:"Santa Fe",86:"Santiago del Estero",
        90:"Tucumán",94:"Tierra del Fuego"}

d18 = df[df.anio == 2018].copy()
print(f"Muestra 2018: {len(d18):,} personas")

def prev(d, col):
    """Prevalencia ponderada con peso_rel (igual que en ENFR_EDA.ipynb)."""
    s = d[d[col].notna()]
    if len(s) == 0 or s.peso_rel.sum() == 0:
        return np.nan, 0
    return float(np.average(s[col], weights=s.peso_rel)), len(s)

## 2 · Media y varianza

En una variable binaria (0/1) **la media es la prevalencia** y la varianza es `p·(1−p)`: no agrega información nueva.
Por eso la dispersión se analiza en las variables continuas, **edad** e **IMC**.

In [ ]:
def cuantil_ponderado(x, w, q):
    orden = np.argsort(x)
    x, w = np.asarray(x)[orden], np.asarray(w)[orden]
    acum = (np.cumsum(w) - 0.5 * w) / w.sum()
    return float(np.interp(q, acum, x))

filas = []
for col, etq in [("edad", "Edad (años)"), ("imc", "IMC (kg/m²)")]:
    s = d18[[col, "peso_rel"]].dropna()
    x, w = s[col].astype(float), s.peso_rel.astype(float)
    media = np.average(x, weights=w)
    var = np.average((x - media) ** 2, weights=w)
    filas.append({"variable": etq, "n": len(s), "media": round(media, 2), "varianza": round(var, 2),
                  "desvio": round(np.sqrt(var), 2), "minimo": round(x.min(), 1),
                  "q1": round(cuantil_ponderado(x, w, .25), 1), "mediana": round(cuantil_ponderado(x, w, .5), 1),
                  "q3": round(cuantil_ponderado(x, w, .75), 1), "maximo": round(x.max(), 1)})
descriptivas = pd.DataFrame(filas)
descriptivas

## 3 · Matriz de correlación

Correlación de Pearson entre las cuatro condiciones (0/1), el IMC y la edad. Entre dos variables binarias
equivale al coeficiente phi. Se usan los pares de datos disponibles en cada caso.

In [ ]:
cols = COND + ["imc", "edad"]
correlacion = d18[cols].astype(float).corr().round(3)
correlacion = correlacion.rename(index=NOMBRE, columns=NOMBRE)
correlacion

**Lectura:** las condiciones se correlacionan en forma positiva y baja (0,1 a 0,2): tienden a darse en las mismas
personas, lo que respalda el hallazgo de la acumulación. La edad es lo más asociado a la hipertensión.
La correlación obesidad–IMC es alta por construcción (obesidad = IMC ≥ 30).

## 4 · Odds ratio de factores asociados a la hipertensión

Para cada factor se arma la tabla 2×2 (factor sí/no × hipertensión sí/no) entre quienes tienen hipertensión conocida.
OR = (a·d)/(b·c), con intervalo de confianza del 95% por el método de Woolf (log OR ± 1,96·EE).
Son **OR crudos**: cada factor se analiza por separado, sin ajustar por los demás.

> **Verificar `AF_BAJO`** contra el diccionario de INDEC: el ETL advierte que el sentido del orden de
> `actividad_fisica` no está confirmado. Si el código está invertido, el OR de baja actividad física también.

In [ ]:
AF_BAJO = 3   # <- código de 'actividad física baja' en nivel_actividad_fisica (VERIFICAR con diccionario INDEC)

h = d18[d18.hta.notna()].copy()
def binf(cond, base):
    """1/0 según cond; NaN donde la variable de origen es NaN."""
    return np.where(base.isna(), np.nan, cond.astype(float))

h["mayor_50"] = binf(h.edad >= 50, h.edad)
h["fumador"] = binf(h.tabaco == 1, h.tabaco)
h["sedentario"] = binf(h.actividad_fisica == AF_BAJO, h.actividad_fisica)
h["sin_cob"] = binf(h.tiene_cobertura == 0, h.tiene_cobertura)
h["q1_q2"] = binf(h.quintil_ingreso <= 2, h.quintil_ingreso)

FACTORES = [("Obesidad (IMC≥30)", "obesidad"), ("Diabetes", "diabetes"), ("Colesterol alto", "colesterol_alto"),
            ("Sobrepeso (IMC≥25)", "sobrepeso"), ("Tener 50 años o más", "mayor_50"), ("Fumador actual", "fumador"),
            ("Baja actividad física", "sedentario"), ("Ser mujer", "mujer"), ("Sin cobertura de salud", "sin_cob"),
            ("Quintil de ingreso 1 o 2", "q1_q2")]

filas = []
for etq, var in FACTORES:
    s = h[[var, "hta"]].dropna()
    a = ((s[var] == 1) & (s.hta == 1)).sum(); b = ((s[var] == 1) & (s.hta == 0)).sum()
    c = ((s[var] == 0) & (s.hta == 1)).sum(); d = ((s[var] == 0) & (s.hta == 0)).sum()
    orr = (a * d) / (b * c)
    ee = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo, hi = np.exp(np.log(orr) - 1.96 * ee), np.exp(np.log(orr) + 1.96 * ee)
    filas.append({"factor": etq, "variable": var, "odds_ratio": round(orr, 3), "ic_bajo": round(lo, 3),
                  "ic_alto": round(hi, 3), "significativo": int(lo > 1 or hi < 1)})
odds = pd.DataFrame(filas)
odds

**Lectura:** edad (50+), diabetes, colesterol alto y obesidad son los factores más fuertes.
Los OR menores a 1 (fumar, sin cobertura) **no indican protección**: con datos transversales y diagnóstico
autorreportado reflejan causalidad inversa (quien recibe el diagnóstico deja de fumar) o sesgo de diagnóstico
(quien no tiene cobertura se controla menos y no sabe que es hipertenso).

## 5 · Prevalencia estandarizada por edad (método directo)

Las provincias tienen distinta estructura de edad, y la hipertensión sube mucho con la edad. Para compararlas
se aplica la prevalencia de cada grupo etario de la provincia a una **población estándar común**: la distribución
de edad del total país 2018 (ponderada).

In [ ]:
base = d18[d18.hta.notna() & d18.grupo_etario.notna()]
estandar = base.groupby("grupo_etario").peso_rel.sum() / base.peso_rel.sum()   # población estándar

filas = []
for cod, nom in PROV.items():
    sub = base[base.provincia == cod]
    cruda, n = prev(d18[d18.provincia == cod], "hta")
    if n < 30:
        continue
    tasas = {g: prev(sub[sub.grupo_etario == g], "hta")[0] for g in estandar.index}
    est = sum(estandar[g] * tasas[g] for g in estandar.index if not np.isnan(tasas[g]))
    est /= sum(estandar[g] for g in estandar.index if not np.isnan(tasas[g]))
    filas.append({"provincia": nom, "prevalencia_cruda": round(cruda, 4), "prevalencia_estandarizada": round(est, 4),
                  "cruda_pct": round(cruda * 100, 1), "estandarizada_pct": round(est * 100, 1),
                  "diferencia_pp": round((est - cruda) * 100, 1), "n": n})
estandarizada = pd.DataFrame(filas).sort_values("prevalencia_estandarizada", ascending=False).reset_index(drop=True)
estandarizada

**Lectura:** el ranking casi no cambia. Formosa sigue primera después de ajustar por edad, así que su prevalencia
alta no se explica por tener una población más envejecida. CABA es la que más baja (−3,2 pp): tiene más
adultos mayores que el promedio.

## 6 · Exportar

In [ ]:
descriptivas.to_csv(os.path.join(SALIDA, "enfr_descriptivas.csv"), index=False, encoding="utf-8")
correlacion.to_csv(os.path.join(SALIDA, "enfr_correlacion.csv"), encoding="utf-8")
odds.to_csv(os.path.join(SALIDA, "enfr_odds_ratio.csv"), index=False, encoding="utf-8")
estandarizada.to_csv(os.path.join(SALIDA, "enfr_estandarizada.csv"), index=False, encoding="utf-8")
print("Guardados en", SALIDA)